In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Carpeta donde se guardará la base del Día 9.
CARPETA_DATOS = Path("datos_dia_09")
CARPETA_DATOS.mkdir(exist_ok=True)

# Archivo SQLite del modelo relacional.
RUTA_DB = CARPETA_DATOS / "modelo_relacional_ventas.db"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB.resolve())

Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_09

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_09\modelo_relacional_ventas.db


In [2]:
def obtener_conexion():
    """
    Crea una conexión con la base de datos SQLite.

    En SQLite, las llaves foráneas deben activarse en cada conexión
    usando PRAGMA foreign_keys = ON.
    """
    conexion = sqlite3.connect(RUTA_DB)
    conexion.execute("PRAGMA foreign_keys = ON")
    return conexion


conexion = obtener_conexion()

estado_fk = pd.read_sql_query("""
PRAGMA foreign_keys
""", conexion)

conexion.close()

estado_fk

,foreign_keys
0,1


In [3]:
conexion = obtener_conexion()
cursor = conexion.cursor()

# Se eliminan primero las tablas hijas y luego las tablas padre.
# Esto evita errores por dependencias de llaves foráneas.
cursor.execute("DROP TABLE IF EXISTS detalle_ventas")
cursor.execute("DROP TABLE IF EXISTS ventas")
cursor.execute("DROP TABLE IF EXISTS productos")
cursor.execute("DROP TABLE IF EXISTS clientes")

cursor.execute("""
CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    telefono TEXT,
    ciudad TEXT DEFAULT 'No especificada'
)
""")

cursor.execute("""
CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    sku TEXT NOT NULL UNIQUE,
    nombre TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio REAL NOT NULL CHECK(precio > 0),
    stock INTEGER NOT NULL DEFAULT 0 CHECK(stock >= 0),
    activo INTEGER NOT NULL DEFAULT 1 CHECK(activo IN (0, 1))
)
""")

cursor.execute("""
CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente INTEGER NOT NULL,
    fecha TEXT NOT NULL,
    total REAL NOT NULL DEFAULT 0 CHECK(total >= 0),

    FOREIGN KEY (id_cliente)
        REFERENCES clientes(id_cliente)
)
""")

cursor.execute("""
CREATE TABLE detalle_ventas (
    id_detalle INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    precio_unitario REAL NOT NULL CHECK(precio_unitario > 0),
    subtotal REAL NOT NULL CHECK(subtotal >= 0),

    FOREIGN KEY (id_venta)
        REFERENCES ventas(id_venta),

    FOREIGN KEY (id_producto)
        REFERENCES productos(id_producto)
)
""")

conexion.commit()
conexion.close()

print("Modelo relacional creado correctamente.")

Modelo relacional creado correctamente.


In [4]:
conexion = obtener_conexion()

tablas = ["clientes", "productos", "ventas", "detalle_ventas"]

for tabla in tablas:
    print(f"Estructura de la tabla: {tabla}")
    display(pd.read_sql_query(f"""
    PRAGMA table_info({tabla})
    """, conexion))

conexion.close()

Estructura de la tabla: clientes


,cid,name,type,notnull,dflt_value,pk
0,0,id_cliente,INTEGER,0,NaN,1
1,1,nombre,TEXT,1,NaN,0
2,2,correo,TEXT,1,NaN,0
3,3,telefono,TEXT,0,NaN,0
4,4,ciudad,TEXT,0,'No especificada',0


Estructura de la tabla: productos


,cid,name,type,notnull,dflt_value,pk
0,0,id_producto,INTEGER,0,NaN,1
1,1,sku,TEXT,1,NaN,0
2,2,nombre,TEXT,1,NaN,0
3,3,categoria,TEXT,1,NaN,0
4,4,precio,REAL,1,NaN,0
5,5,stock,INTEGER,1,0,0
6,6,activo,INTEGER,1,1,0


Estructura de la tabla: ventas


,cid,name,type,notnull,dflt_value,pk
0,0,id_venta,INTEGER,0,NaN,1
1,1,id_cliente,INTEGER,1,NaN,0
2,2,fecha,TEXT,1,NaN,0
3,3,total,REAL,1,0,0


Estructura de la tabla: detalle_ventas


,cid,name,type,notnull,dflt_value,pk
0,0,id_detalle,INTEGER,0,None,1
1,1,id_venta,INTEGER,1,None,0
2,2,id_producto,INTEGER,1,None,0
3,3,cantidad,INTEGER,1,None,0
4,4,precio_unitario,REAL,1,None,0
5,5,subtotal,REAL,1,None,0


In [6]:
conexion = obtener_conexion()

tablas_con_fk = ["ventas", "detalle_ventas"]

for tabla in tablas_con_fk:
    print(f"Llaves foráneas de la tabla: {tabla}")
    display(pd.read_sql_query(f"""
    PRAGMA foreign_key_list({tabla})
    """, conexion))

conexion.close()

Llaves foráneas de la tabla: ventas


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,clientes,id_cliente,id_cliente,NO ACTION,NO ACTION,NONE


Llaves foráneas de la tabla: detalle_ventas


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,productos,id_producto,id_producto,NO ACTION,NO ACTION,NONE
1,1,0,ventas,id_venta,id_venta,NO ACTION,NO ACTION,NONE


In [7]:
conexion = obtener_conexion()
cursor = conexion.cursor()

clientes = [
    ("Ana López", "ana@example.com", "555-111-2222", "CDMX"),
    ("Carlos Pérez", "carlos@example.com", "555-333-4444", "Guadalajara"),
    ("María Torres", "maria@example.com", None, "Monterrey"),
    ("Luis Romero", "luis@example.com", "555-888-9999", "Puebla")
]

productos = [
    ("P001", "Mouse inalámbrico", "Accesorios", 249.90, 15, 1),
    ("P002", "Teclado mecánico", "Accesorios", 899.00, 8, 1),
    ("P003", "Monitor 24 pulgadas", "Pantallas", 3299.00, 4, 1),
    ("P004", "Cable HDMI", "Cables", 120.00, 20, 1),
    ("P005", "Memoria USB 64GB", "Almacenamiento", 150.00, 30, 1),
    ("P006", "Bocinas Bluetooth", "Audio", 699.00, 0, 1)
]

cursor.executemany("""
INSERT INTO clientes (nombre, correo, telefono, ciudad)
VALUES (?, ?, ?, ?)
""", clientes)

cursor.executemany("""
INSERT INTO productos (sku, nombre, categoria, precio, stock, activo)
VALUES (?, ?, ?, ?, ?, ?)
""", productos)

conexion.commit()
conexion.close()

print("Clientes y productos insertados correctamente.")

Clientes y productos insertados correctamente.


In [8]:
conexion = obtener_conexion()

print("Tabla clientes")
display(pd.read_sql_query("""
SELECT *
FROM clientes
ORDER BY id_cliente
""", conexion))

print("Tabla productos")
display(pd.read_sql_query("""
SELECT *
FROM productos
ORDER BY id_producto
""", conexion))

conexion.close()

Tabla clientes


,id_cliente,nombre,correo,telefono,ciudad
0,1,Ana López,ana@example.com,555-111-2222,CDMX
1,2,Carlos Pérez,carlos@example.com,555-333-4444,Guadalajara
2,3,María Torres,maria@example.com,NaN,Monterrey
3,4,Luis Romero,luis@example.com,555-888-9999,Puebla


Tabla productos


,id_producto,sku,nombre,categoria,precio,stock,activo
0,1,P001,Mouse inalámbrico,Accesorios,249.9,15,1
1,2,P002,Teclado mecánico,Accesorios,899.0,8,1
2,3,P003,Monitor 24 pulgadas,Pantallas,3299.0,4,1
3,4,P004,Cable HDMI,Cables,120.0,20,1
4,5,P005,Memoria USB 64GB,Almacenamiento,150.0,30,1
5,6,P006,Bocinas Bluetooth,Audio,699.0,0,1


In [9]:
def registrar_venta(id_cliente, fecha, productos_vendidos):
    """
    Registra una venta completa.

    id_cliente:
        Cliente que realiza la compra.

    fecha:
        Fecha de la venta.

    productos_vendidos:
        Lista de diccionarios. Cada diccionario debe incluir:
        id_producto y cantidad.

    La función:
        1. Crea el encabezado de la venta.
        2. Registra cada producto vendido en detalle_ventas.
        3. Calcula subtotales.
        4. Actualiza el total de la venta.
        5. Descuenta stock del producto.
    """
    conexion = obtener_conexion()
    cursor = conexion.cursor()

    try:
        # Crear encabezado de venta con total inicial en 0.
        cursor.execute("""
        INSERT INTO ventas (id_cliente, fecha, total)
        VALUES (?, ?, ?)
        """, (id_cliente, fecha, 0))

        id_venta = cursor.lastrowid
        total_venta = 0

        for item in productos_vendidos:
            id_producto = item["id_producto"]
            cantidad = item["cantidad"]

            # Consultar precio y stock actual del producto.
            cursor.execute("""
            SELECT precio, stock
            FROM productos
            WHERE id_producto = ? AND activo = 1
            """, (id_producto,))

            producto = cursor.fetchone()

            if producto is None:
                raise ValueError(f"El producto con id {id_producto} no existe o no está activo.")

            precio_unitario, stock_actual = producto

            if stock_actual < cantidad:
                raise ValueError(f"No hay stock suficiente para el producto {id_producto}.")

            subtotal = precio_unitario * cantidad
            total_venta += subtotal

            # Insertar renglón en detalle_ventas.
            cursor.execute("""
            INSERT INTO detalle_ventas (
                id_venta,
                id_producto,
                cantidad,
                precio_unitario,
                subtotal
            )
            VALUES (?, ?, ?, ?, ?)
            """, (id_venta, id_producto, cantidad, precio_unitario, subtotal))

            # Descontar stock.
            cursor.execute("""
            UPDATE productos
            SET stock = stock - ?
            WHERE id_producto = ?
            """, (cantidad, id_producto))

        # Actualizar total de la venta.
        cursor.execute("""
        UPDATE ventas
        SET total = ?
        WHERE id_venta = ?
        """, (total_venta, id_venta))

        conexion.commit()

        print(f"Venta registrada correctamente. ID venta: {id_venta}")
        print(f"Total de la venta: ${total_venta:.2f}")

    except Exception as error:
        conexion.rollback()
        print("No se pudo registrar la venta.")
        print("Motivo:", error)

    finally:
        conexion.close()

In [10]:
registrar_venta(
    id_cliente=1,
    fecha="2026-06-25",
    productos_vendidos=[
        {"id_producto": 1, "cantidad": 2},
        {"id_producto": 2, "cantidad": 1}
    ]
)

registrar_venta(
    id_cliente=2,
    fecha="2026-06-26",
    productos_vendidos=[
        {"id_producto": 3, "cantidad": 1},
        {"id_producto": 4, "cantidad": 2}
    ]
)

registrar_venta(
    id_cliente=1,
    fecha="2026-06-27",
    productos_vendidos=[
        {"id_producto": 5, "cantidad": 3},
        {"id_producto": 4, "cantidad": 1}
    ]
)

Venta registrada correctamente. ID venta: 1
Total de la venta: $1398.80
Venta registrada correctamente. ID venta: 2
Total de la venta: $3539.00
Venta registrada correctamente. ID venta: 3
Total de la venta: $570.00


In [11]:
conexion = obtener_conexion()

print("Ventas")
display(pd.read_sql_query("""
SELECT *
FROM ventas
ORDER BY id_venta
""", conexion))

print("Detalle de ventas")
display(pd.read_sql_query("""
SELECT *
FROM detalle_ventas
ORDER BY id_detalle
""", conexion))

print("Productos con stock actualizado")
display(pd.read_sql_query("""
SELECT id_producto, sku, nombre, stock
FROM productos
ORDER BY id_producto
""", conexion))

conexion.close()

Ventas


,id_venta,id_cliente,fecha,total
0,1,1,2026-06-25,1398.8
1,2,2,2026-06-26,3539.0
2,3,1,2026-06-27,570.0


Detalle de ventas


,id_detalle,id_venta,id_producto,cantidad,precio_unitario,subtotal
0,1,1,1,2,249.9,499.8
1,2,1,2,1,899.0,899.0
2,3,2,3,1,3299.0,3299.0
3,4,2,4,2,120.0,240.0
4,5,3,5,3,150.0,450.0
5,6,3,4,1,120.0,120.0


Productos con stock actualizado


,id_producto,sku,nombre,stock
0,1,P001,Mouse inalámbrico,13
1,2,P002,Teclado mecánico,7
2,3,P003,Monitor 24 pulgadas,3
3,4,P004,Cable HDMI,17
4,5,P005,Memoria USB 64GB,27
5,6,P006,Bocinas Bluetooth,0


In [12]:
conexion = obtener_conexion()

ventas_con_cliente = pd.read_sql_query("""
SELECT
    ventas.id_venta,
    ventas.fecha,
    ventas.id_cliente,
    clientes.nombre AS cliente,
    clientes.correo,
    ventas.total
FROM ventas
INNER JOIN clientes
    ON ventas.id_cliente = clientes.id_cliente
ORDER BY ventas.id_venta
""", conexion)

conexion.close()

ventas_con_cliente

,id_venta,fecha,id_cliente,cliente,correo,total
0,1,2026-06-25,1,Ana López,ana@example.com,1398.8
1,2,2026-06-26,2,Carlos Pérez,carlos@example.com,3539.0
2,3,2026-06-27,1,Ana López,ana@example.com,570.0


In [13]:
conexion = obtener_conexion()

ventas_con_cliente_alias = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    c.correo,
    v.total
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
ORDER BY v.id_venta
""", conexion)

conexion.close()

ventas_con_cliente_alias

,id_venta,fecha,cliente,correo,total
0,1,2026-06-25,Ana López,ana@example.com,1398.8
1,2,2026-06-26,Carlos Pérez,carlos@example.com,3539.0
2,3,2026-06-27,Ana López,ana@example.com,570.0


In [14]:
conexion = obtener_conexion()

detalle_con_producto = pd.read_sql_query("""
SELECT
    dv.id_detalle,
    dv.id_venta,
    p.sku,
    p.nombre AS producto,
    p.categoria,
    dv.cantidad,
    dv.precio_unitario,
    dv.subtotal
FROM detalle_ventas dv
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
ORDER BY dv.id_venta, dv.id_detalle
""", conexion)

conexion.close()

detalle_con_producto

,id_detalle,id_venta,sku,producto,categoria,cantidad,precio_unitario,subtotal
0,1,1,P001,Mouse inalámbrico,Accesorios,2,249.9,499.8
1,2,1,P002,Teclado mecánico,Accesorios,1,899.0,899.0
2,3,2,P003,Monitor 24 pulgadas,Pantallas,1,3299.0,3299.0
3,4,2,P004,Cable HDMI,Cables,2,120.0,240.0
4,5,3,P005,Memoria USB 64GB,Almacenamiento,3,150.0,450.0
5,6,3,P004,Cable HDMI,Cables,1,120.0,120.0


In [15]:
conexion = obtener_conexion()

reporte_ventas = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    c.correo,
    p.sku,
    p.nombre AS producto,
    p.categoria,
    dv.cantidad,
    dv.precio_unitario,
    dv.subtotal,
    v.total AS total_venta
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
ORDER BY v.id_venta, dv.id_detalle
""", conexion)

conexion.close()

reporte_ventas

,id_venta,fecha,cliente,correo,sku,producto,categoria,cantidad,precio_unitario,subtotal,total_venta
0,1,2026-06-25,Ana López,ana@example.com,P001,Mouse inalámbrico,Accesorios,2,249.9,499.8,1398.8
1,1,2026-06-25,Ana López,ana@example.com,P002,Teclado mecánico,Accesorios,1,899.0,899.0,1398.8
2,2,2026-06-26,Carlos Pérez,carlos@example.com,P003,Monitor 24 pulgadas,Pantallas,1,3299.0,3299.0,3539.0
3,2,2026-06-26,Carlos Pérez,carlos@example.com,P004,Cable HDMI,Cables,2,120.0,240.0,3539.0
4,3,2026-06-27,Ana López,ana@example.com,P005,Memoria USB 64GB,Almacenamiento,3,150.0,450.0,570.0
5,3,2026-06-27,Ana López,ana@example.com,P004,Cable HDMI,Cables,1,120.0,120.0,570.0


In [16]:
conexion = obtener_conexion()

clientes_con_ventas_inner = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre AS cliente,
    v.id_venta,
    v.fecha,
    v.total
FROM clientes c
INNER JOIN ventas v
    ON c.id_cliente = v.id_cliente
ORDER BY c.id_cliente, v.id_venta
""", conexion)

conexion.close()

clientes_con_ventas_inner

,id_cliente,cliente,id_venta,fecha,total
0,1,Ana López,1,2026-06-25,1398.8
1,1,Ana López,3,2026-06-27,570.0
2,2,Carlos Pérez,2,2026-06-26,3539.0


In [17]:
conexion = obtener_conexion()

clientes_con_ventas_left = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre AS cliente,
    v.id_venta,
    v.fecha,
    v.total
FROM clientes c
LEFT JOIN ventas v
    ON c.id_cliente = v.id_cliente
ORDER BY c.id_cliente, v.id_venta
""", conexion)

conexion.close()

clientes_con_ventas_left

,id_cliente,cliente,id_venta,fecha,total
0,1,Ana López,1.0,2026-06-25,1398.8
1,1,Ana López,3.0,2026-06-27,570.0
2,2,Carlos Pérez,2.0,2026-06-26,3539.0
3,3,María Torres,NaN,NaN,NaN
4,4,Luis Romero,NaN,NaN,NaN


In [18]:
conexion = obtener_conexion()

clientes_sin_ventas = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre,
    c.correo
FROM clientes c
LEFT JOIN ventas v
    ON c.id_cliente = v.id_cliente
WHERE v.id_venta IS NULL
ORDER BY c.id_cliente
""", conexion)

conexion.close()

clientes_sin_ventas

,id_cliente,nombre,correo
0,3,María Torres,maria@example.com
1,4,Luis Romero,luis@example.com


In [19]:
conexion = obtener_conexion()

clientes_sin_ventas = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre,
    c.correo
FROM clientes c
LEFT JOIN ventas v
    ON c.id_cliente = v.id_cliente
WHERE v.id_venta IS NULL
ORDER BY c.id_cliente
""", conexion)

conexion.close()

clientes_sin_ventas

,id_cliente,nombre,correo
0,3,María Torres,maria@example.com
1,4,Luis Romero,luis@example.com


In [20]:
conexion = obtener_conexion()

todos_productos_ventas = pd.read_sql_query("""
SELECT
    p.id_producto,
    p.sku,
    p.nombre AS producto,
    COALESCE(SUM(dv.cantidad), 0) AS unidades_vendidas,
    COALESCE(SUM(dv.subtotal), 0) AS importe_vendido
FROM productos p
LEFT JOIN detalle_ventas dv
    ON p.id_producto = dv.id_producto
GROUP BY p.id_producto, p.sku, p.nombre
ORDER BY importe_vendido DESC
""", conexion)

conexion.close()

todos_productos_ventas

,id_producto,sku,producto,unidades_vendidas,importe_vendido
0,3,P003,Monitor 24 pulgadas,1,3299.0
1,2,P002,Teclado mecánico,1,899.0
2,1,P001,Mouse inalámbrico,2,499.8
3,5,P005,Memoria USB 64GB,3,450.0
4,4,P004,Cable HDMI,3,360.0
5,6,P006,Bocinas Bluetooth,0,0.0


In [21]:
conexion = obtener_conexion()

productos_no_vendidos = pd.read_sql_query("""
SELECT
    p.id_producto,
    p.sku,
    p.nombre AS producto,
    p.stock
FROM productos p
LEFT JOIN detalle_ventas dv
    ON p.id_producto = dv.id_producto
WHERE dv.id_detalle IS NULL
ORDER BY p.id_producto
""", conexion)

conexion.close()

productos_no_vendidos

,id_producto,sku,producto,stock
0,6,P006,Bocinas Bluetooth,0


In [22]:
conexion = obtener_conexion()
cursor = conexion.cursor()

try:
    cursor.execute("""
    INSERT INTO ventas (id_cliente, fecha, total)
    VALUES (?, ?, ?)
    """, (999, "2026-06-30", 0))

    conexion.commit()
    print("Venta registrada.")

except sqlite3.IntegrityError as error:
    print("No se pudo registrar la venta.")
    print("Motivo:", error)

finally:
    conexion.close()

No se pudo registrar la venta.
Motivo: FOREIGN KEY constraint failed


In [23]:
conexion = obtener_conexion()
cursor = conexion.cursor()

try:
    cursor.execute("""
    INSERT INTO detalle_ventas (
        id_venta,
        id_producto,
        cantidad,
        precio_unitario,
        subtotal
    )
    VALUES (?, ?, ?, ?, ?)
    """, (1, 999, 1, 100.00, 100.00))

    conexion.commit()
    print("Detalle registrado.")

except sqlite3.IntegrityError as error:
    print("No se pudo registrar el detalle.")
    print("Motivo:", error)

finally:
    conexion.close()

No se pudo registrar el detalle.
Motivo: FOREIGN KEY constraint failed


In [24]:
conexion = obtener_conexion()

reporte_reconstruido = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    p.nombre AS producto,
    dv.cantidad,
    dv.precio_unitario,
    dv.subtotal,
    v.total
FROM ventas v
INNER JOIN clientes c
    ON v.id_cliente = c.id_cliente
INNER JOIN detalle_ventas dv
    ON v.id_venta = dv.id_venta
INNER JOIN productos p
    ON dv.id_producto = p.id_producto
ORDER BY v.id_venta, dv.id_detalle
""", conexion)

conexion.close()

reporte_reconstruido

,id_venta,fecha,cliente,producto,cantidad,precio_unitario,subtotal,total
0,1,2026-06-25,Ana López,Mouse inalámbrico,2,249.9,499.8,1398.8
1,1,2026-06-25,Ana López,Teclado mecánico,1,899.0,899.0,1398.8
2,2,2026-06-26,Carlos Pérez,Monitor 24 pulgadas,1,3299.0,3299.0,3539.0
3,2,2026-06-26,Carlos Pérez,Cable HDMI,2,120.0,240.0,3539.0
4,3,2026-06-27,Ana López,Memoria USB 64GB,3,150.0,450.0,570.0
5,3,2026-06-27,Ana López,Cable HDMI,1,120.0,120.0,570.0
